In [2]:
import pandas as pd
import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

### Combine all CSVs in CICIoT-2023 folder

In [ ]:
csv_files = sorted(glob.glob(f"./Datasets/CICIoT-2023/*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in Datasets/CICIoT-2023/")

# Read first file keeping header, then read remaining files skipping their header rows
first_df = pd.read_csv(csv_files[0])
rest_dfs = []
for f in csv_files[1:]:
    # skip the header row of subsequent files and assign the same column names as the first file
    df = pd.read_csv(f, skiprows=1, header=None, names=first_df.columns)
    rest_dfs.append(df)


In [ ]:
combined = pd.concat([first_df] + rest_dfs, ignore_index=True)

out_path = f"./Datasets/merged_combined_final.csv"
combined.to_csv(out_path, index=False)
print(f"Combined {len(csv_files)} files into: {out_path}; shape={combined.shape}")

### Read Dataset

In [ ]:
df = pd.read_csv(f"./Datasets/merged_combined_final.csv")

#### Just to get an idea of what the combined dataset contains

In [ ]:
print(df.shape)

In [ ]:
print(df.head())

### Cleaning

#### Removing NaN values and Duplicates

In [ ]:
df.dropna()

In [ ]:
df.drop_duplicates()

In [ ]:
print(df.shape)

#### Change every header to lowercase

In [ ]:
df.columns = df.columns.str.lower()

#### Drop unneeded values

In [ ]:
df = df[df['label'].str.contains('DDOS|MIRAI|BENIGN', case=False)]

#### Label encoder

In [ ]:
le = LabelEncoder()

In [ ]:
df['label_encoded'] = df['label'].apply(lambda x: 0 if 'BENIGN' in x else 1)

#### Save cleaned file

In [ ]:
df.to_csv("./Datasets/final/cleaned-encoded.csv", index=False)

#### Just to get an idea of what the filtered dataset contains

In [ ]:
print(df.shape)

In [ ]:
print(df.head())

In [ ]:
print(df['label_encoded'].unique()) 

In [ ]:
print(df['label_encoded'].value_counts())

### Sampling

#### Stratified Sampling

In [3]:
df = pd.read_csv(f"./Datasets/final/cleaned-encoded.csv")

In [4]:
ddos_count = df[df['label'].str.startswith('DDOS')].shape[0]
mirai_count = df[df['label'].str.startswith('MIRAI')].shape[0]
benign_count = df[df['label'].str.startswith('BENIGN')].shape[0]

print("DDOS:", ddos_count)
print("MIRAI:", mirai_count)
print("BENIGN:", benign_count)

DDOS: 5281157
MIRAI: 409431
BENIGN: 170496


In [4]:
df_sample, _ = train_test_split(
    df,
    test_size=0.5,                # keeps 50% as sample
    stratify=df['label_encoded'], # ensures same class distribution
    random_state=42               # for reproducibility
)

##### Save sample to csv

In [5]:
df_sample.to_csv("./Datasets/final/sample-cleaned-encoded.csv", index=False)

In [ ]:
df_sample.head()

In [6]:
print(df_sample.shape)

(2930542, 41)
